# 🏀 NBA Lakehouse — Notebook 3: Silver → Gold

## Overview
This notebook implements the **third and final layer** of the Medallion Architecture: transforming clean Silver tables into **analytics-ready Gold tables** optimized for Power BI reporting.

## Architecture
```
Silver (cleaned Delta)  →  Gold (analytics-ready Delta)
  Individual tables          Joined, aggregated, enriched
```

## Gold Tables Built
| Table | Source Tables | Purpose | Rows |
|-------|--------------|---------|------|
| `player_season_summary` | player_per_game + advanced | Complete player stats per season | 16,565 |
| `team_performance_agg` | team_summaries | Team win %, ratings, playoffs | 805 |
| `clutch_scoring_analysis` | player_per_game + advanced | Scoring efficiency & clutch metrics | 7,190 |

## Design Decisions
- **Joins** use `left` strategy so players without advanced stats are still included
- **`player_season_summary`** joins on 4 keys (`player`, `player_id`, `season`, `team`) to avoid cross-season/cross-team duplicates
- **`clutch_scoring_analysis`** filters to players with ≥20 MPG and ≥20 games — removes bench players who inflate efficiency on small samples
- **`scoring_efficiency`** is a custom metric: `(pts_per_game × fg_percent) + (ast_per_game × 0.5)` — rewards high-volume efficient scorers who also create for teammates
- **`all_around_score`** is a simple sum of pts + ast + reb + stl + blk — a quick "impact" proxy

## Step 1 — Imports

In [0]:
# PySpark SQL functions used across all Gold transformations
from pyspark.sql.functions import (
    col,
    round as spark_round,   # renamed to avoid conflict with Python built-in round()
    avg, sum, max, min,     # aggregation functions (used in future extensions)
    count, desc, rank       # sorting and ranking functions
)
from pyspark.sql.window import Window  # for window functions (ranking within groups)

print("✓ Imports ready!")

## Step 2 — Gold: player_season_summary

This table is the **core player analytics table** — it joins per-game stats with advanced metrics to give a complete view of each player's season.

**Join strategy:**
- Join key: `player`, `player_id`, `season`, `team` (4 columns)
- Using all 4 keys prevents false matches for players who switched teams mid-season or share similar names
- `left` join ensures players without advanced stats (rare edge cases) are still included

**Key columns selected:**
- Identity: `season`, `player`, `player_id`, `team`, `pos`, `age`
- Volume: `g`, `mp_per_game`
- Basic stats: `pts_per_game`, `ast_per_game`, `trb_per_game`, `stl_per_game`, `blk_per_game`
- Shooting: `fg_percent`, `x3p_percent`, `ft_percent`
- Advanced: `per`, `ws`, `ws_48`, `bpm`, `vorp`, `usg_percent`

In [0]:
# Read cleaned Silver tables
df_ppg = spark.read.table("nba_lakehouse_catalog.silver.player_per_game")
df_adv = spark.read.table("nba_lakehouse_catalog.silver.advanced")

# Select only the advanced columns we need to avoid column name conflicts
# and reduce shuffle size during the join
df_gold = df_ppg.join(
    df_adv.select("player", "player_id", "season", "team",
                  "per", "ws", "ws_48", "bpm", "vorp", "usg_percent"),
    on=["player", "player_id", "season", "team"],  # 4-key join prevents duplicates
    how="left"  # keep all player_per_game rows even if no advanced match
)

# Select final columns for the Gold table — ordered logically for readability
df_gold = df_gold.select(
    # Identity fields
    "season", "player", "player_id", "team", "pos", "age",
    # Availability
    "g", "mp_per_game",
    # Core stats
    "pts_per_game", "ast_per_game", "trb_per_game",
    "stl_per_game", "blk_per_game",
    # Shooting efficiency
    "fg_percent", "x3p_percent", "ft_percent",
    # Advanced metrics
    "per", "ws", "ws_48", "bpm", "vorp", "usg_percent"
)

# Write to Gold as a managed Delta table
df_gold.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.gold.player_season_summary")
print(f"✓ gold.player_season_summary — {df_gold.count()} rows written")

## Step 3 — Gold: team_performance_agg

This table powers the **Team Performance Analysis** dashboard page. It contains one row per team per season with all key performance indicators.

**Columns included:**
- `w`, `l`, `win_pct` — win/loss record and win percentage
- `o_rtg`, `d_rtg`, `n_rtg` — offensive, defensive, and net ratings (points per 100 possessions)
- `pace` — possessions per 48 minutes
- `srs` — Simple Rating System (point differential adjusted for schedule)
- `mov` — margin of victory
- `playoffs` — boolean flag for playoff appearance
- `ts_percent` — team true shooting percentage
- `attend_g` — average attendance per game

**Ordering:** Sorted by season and win_pct descending — best teams appear first within each season.

In [0]:
# Read clean team summaries from Silver
df = spark.read.table("nba_lakehouse_catalog.silver.team_summaries")

# Select key performance columns — no aggregation needed
# Silver already has one row per team per season
df_gold = df.select(
    # Identity
    "season", "team", "abbreviation",
    # Win/loss record
    "w", "l", "win_pct",
    # Four Factors ratings (Dean Oliver's framework)
    "o_rtg", "d_rtg", "n_rtg",
    # Style and dominance metrics
    "pace", "srs", "mov",
    # Context
    "playoffs", "ts_percent", "attend_g"
).orderBy(
    "season",           # chronological order
    desc("win_pct")     # best teams first within each season
)

df_gold.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.gold.team_performance_agg")
print(f"✓ gold.team_performance_agg — {df_gold.count()} rows written")

## Step 4 — Gold: clutch_scoring_analysis

This table powers the **Clutch Scoring & Efficiency Analysis** dashboard page. It focuses on meaningful contributors (not bench players) and includes two custom engineered features.

**Filtering:** Only players with `mp_per_game >= 20` AND `g >= 20`
- Removes players with inflated efficiency on tiny sample sizes
- Ensures we're analyzing real contributors, not garbage time players

**Custom metrics:**

1. **`scoring_efficiency`** = `(pts_per_game × fg_percent) + (ast_per_game × 0.5)`
   - Rewards players who score efficiently (high FG%) AND create for teammates
   - A player averaging 30pts on 30% FG scores lower than one averaging 25pts on 55% FG
   - The 0.5 multiplier for assists reflects that not every assist leads to a made basket

2. **`all_around_score`** = `pts + ast + reb + stl + blk`
   - Simple "box score impact" metric
   - Highlights two-way players who contribute across multiple statistical categories

In [0]:
# Read Silver tables for the join
df_ppg = spark.read.table("nba_lakehouse_catalog.silver.player_per_game")
df_adv = spark.read.table("nba_lakehouse_catalog.silver.advanced")

# Join per-game stats with advanced metrics
# Select only the advanced columns needed to keep the join lightweight
df = df_ppg.join(
    df_adv.select("player", "player_id", "season", "team", "per", "ws", "bpm", "vorp"),
    on=["player", "player_id", "season", "team"],
    how="left"
)

# Filter to meaningful contributors only
# mp_per_game >= 20: at least 20 minutes per game (starter/key rotation player)
# g >= 20: at least 20 games played (avoids injury-limited small samples)
df = df.filter((col("mp_per_game") >= 20) & (col("g") >= 20))

# Feature engineering: Scoring Efficiency
# Formula: (points × shooting_percentage) + (assists × 0.5)
# Logic: rewards efficient scorers who also create for teammates
df_gold = df.withColumn(
    "scoring_efficiency",
    spark_round(
        (col("pts_per_game") * col("fg_percent")) + (col("ast_per_game") * 0.5),
        3  # round to 3 decimal places
    )
).withColumn(
    # Feature engineering: All-Around Score
    # Simple sum of major statistical categories — highlights complete players
    "all_around_score",
    spark_round(
        col("pts_per_game") + col("ast_per_game") + col("trb_per_game") +
        col("stl_per_game") + col("blk_per_game"),
        2
    )
)

# Select final columns and sort by scoring efficiency descending
# Most efficient scorers appear first
df_gold = df_gold.select(
    # Identity
    "season", "player", "player_id", "team", "pos", "age",
    # Volume context
    "g", "mp_per_game",
    # Raw production
    "pts_per_game", "ast_per_game", "trb_per_game",
    "stl_per_game", "blk_per_game",
    # Efficiency
    "fg_percent", "ft_percent",
    # Advanced
    "per", "bpm", "vorp",
    # Custom engineered features
    "scoring_efficiency", "all_around_score"
).orderBy(desc("scoring_efficiency"))

df_gold.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.gold.clutch_scoring_analysis")
print(f"✓ gold.clutch_scoring_analysis — {df_gold.count()} rows written")

## Step 5 — Verify Gold Layer + Preview Top 10

In [0]:
# Verify all 3 Gold tables were written with expected dimensions
print("=== Gold Layer Summary ===\n")

gold_tables = ["player_season_summary", "team_performance_agg", "clutch_scoring_analysis"]

for t in gold_tables:
    df = spark.read.table(f"nba_lakehouse_catalog.gold.{t}")
    print(f"✓ {t}: {df.count()} rows | {len(df.columns)} cols")

# Preview the Top 10 most efficient scorers of all time (2000–2026)
# This is the headline insight from the Gold layer
print("\n=== Top 10 Players by Scoring Efficiency (All Time) ===")
df = spark.read.table("nba_lakehouse_catalog.gold.clutch_scoring_analysis")

display(
    df.select("season", "player", "team", "pts_per_game", "fg_percent", "per", "scoring_efficiency")
    .orderBy(desc("scoring_efficiency"))
    .limit(10)
)

## Summary

✅ **Gold layer complete** — 3 analytics-ready Delta tables registered in `nba_lakehouse_catalog.gold`

| Table | Rows | Cols | Powers |
|-------|------|------|--------|
| `player_season_summary` | 16,565 | 22 | Page 1 — Player Performance |
| `team_performance_agg` | 805 | 15 | Page 2 — Team Performance |
| `clutch_scoring_analysis` | 7,190 | 20 | Page 3 — Clutch Scoring |

**Key findings:**
- 🏆 Most efficient scorer: **Nikola Jokić** (2025 season, efficiency = 22.15)
- 🏆 Winningest franchise: **San Antonio Spurs** (avg win_pct ~0.60 since 2000)
- 🏆 Highest single-season PPG: **36.10** (Kobe Bryant, 2005-06)

**Next:** Connect Power BI Desktop to Databricks via the Serverless SQL Warehouse to build the interactive dashboard.